In [1]:
import duckdb
import pandas as pd

## Baloto

In [2]:
path = r'/home/sebas/projects/opensource/langgraph/resources/baloto.csv'

In [3]:
df = pd.read_csv(path)
df.head()

,Date,C1,C2,C3,C4,C5,SB
0,4/22/2017,15,28,36,37,41,10
1,4/26/2017,1,2,25,31,33,14
2,4/29/2017,6,19,25,28,36,1
3,5/3/2017,5,6,7,12,37,6
4,5/6/2017,1,20,36,39,43,10


In [4]:
results = duckdb.sql("SELECT * FROM df").df()
results.tail()

,Date,C1,C2,C3,C4,C5,SB
598,2/22/2023,13,14,6,8,22,12
599,2/25/2023,32,35,14,20,12,9
600,3/1/2023,16,36,39,42,23,11
601,3/4/2023,5,33,4,12,24,12
602,3/8/2023,26,41,35,24,36,7


Suma de la columna C1

In [5]:
query = '''
WITH db as (
SELECT strptime(date, '%m/%d/%Y')::DATE AS fecha,
C1,
C2,
C3,
C4,
C5,
SB
FROM results
)

SELECT SUM(C1) 
FROM DB
WHERE fecha >= '2023-01-01'
'''

duckdb.sql(query)


┌─────────┐
│ sum(C1) │
│ int128  │
├─────────┤
│     348 │
└─────────┘

## Covid 19

In [6]:
covid_path = r'/home/sebas/projects/opensource/langgraph/resources/covid_19_clean_complete.csv'

In [7]:
df_covid = pd.read_csv(covid_path)
df_covid.head()

,Province/State,Country/Region,Lat,Long,Date,Confirmed,Deaths,Recovered,Active,WHO Region
0,NaN,Afghanistan,33.93911,67.709953,2020-01-22,0,0,0,0,Eastern Mediterranean
1,NaN,Albania,41.15330,20.168300,2020-01-22,0,0,0,0,Europe
2,NaN,Algeria,28.03390,1.659600,2020-01-22,0,0,0,0,Africa
3,NaN,Andorra,42.50630,1.521800,2020-01-22,0,0,0,0,Europe
4,NaN,Angola,-11.20270,17.873900,2020-01-22,0,0,0,0,Africa


In [8]:
covid_results = duckdb.sql("SELECT * FROM df_covid")
covid_results

┌──────────────────────────────┬─────────────────────┬──────────┬───────────┬────────────┬───────────┬────────┬───────────┬────────┬───────────────────────┐
│        Province/State        │   Country/Region    │   Lat    │   Long    │    Date    │ Confirmed │ Deaths │ Recovered │ Active │      WHO Region       │
│           varchar            │       varchar       │  double  │  double   │  varchar   │   int64   │ int64  │   int64   │ int64  │        varchar        │
├──────────────────────────────┼─────────────────────┼──────────┼───────────┼────────────┼───────────┼────────┼───────────┼────────┼───────────────────────┤
│ NULL                         │ Afghanistan         │ 33.93911 │ 67.709953 │ 2020-01-22 │         0 │      0 │         0 │      0 │ Eastern Mediterranean │
│ NULL                         │ Albania             │  41.1533 │   20.1683 │ 2020-01-22 │         0 │      0 │         0 │      0 │ Europe                │
│ NULL                         │ Algeria             │  28

Dia en el cual hubo mas casos de covid 19 confirmados y en que pais fue

In [9]:
covid_query = '''
WITH db AS (
    SELECT
        strptime(Date, '%Y-%m-%d')::DATE AS fecha,
        "Country/Region",
        SUM(Confirmed) AS casos
    FROM covid_results
    GROUP BY 1, 2
)

SELECT 
fecha, 
"Country/Region", 
casos
FROM db
WHERE fecha >= '2020-07-12'
ORDER BY casos DESC
LIMIT 1
'''

duckdb.sql(covid_query)

┌────────────┬────────────────┬─────────┐
│   fecha    │ Country/Region │  casos  │
│    date    │    varchar     │ int128  │
├────────────┼────────────────┼─────────┤
│ 2020-07-27 │ US             │ 4290259 │
└────────────┴────────────────┴─────────┘